# SPY Risk Management Under Instability: A 60/40 Tactical Sleeve

**Goal.** Build a no-lookahead automatic risk-management stack that keeps **60% in SPY at all times** and uses a **40% tactical sleeve** only during extreme risk-off windows. The tactical sleeve is evaluated through three lenses: a Markov transition model, a Bayesian posterior model, and a walk-forward ML model.

**Why this notebook exists**
1. SPY has a strong positive drift, so naive de-risking often destroys long-run return.
2. Panic is not the same thing as a short signal. In the previous SPY/VIX study, severe stress often behaved more like a future rebound setup than a continuation setup.
3. A Medallion-style mindset means *small edges, strict leakage controls, and regime-specific capital allocation*, not one heroic forecast.

**Research questions**
1. Can we detect extreme risk-off regimes from current and trailing market plus macro context without lookahead?
2. Inside those regimes, is the next move more likely to be continuation or rebound?
3. Does a 60% core plus 40% tactical sleeve beat simpler heuristics on instability-adjusted performance?
4. Which modelling family looks most sound: Markov transitions, Bayesian shrinkage, or walk-forward ML?


## 1. Load the research outputs

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})
sns.set_theme(style='whitegrid', context='notebook')
pd.options.display.float_format = '{:,.4f}'.format

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESEARCH_DIR = PROJECT_ROOT / 'outputs' / 'spy_regime_risk_management'
print('research dir:', RESEARCH_DIR)

In [ ]:
with open(RESEARCH_DIR / 'spy_regime_risk_management_summary.json', encoding='utf-8') as f:
    summary = json.load(f)

drift = pd.read_csv(RESEARCH_DIR / 'drift_summary.csv')
events = pd.read_csv(RESEARCH_DIR / 'extreme_risk_off_events.csv', parse_dates=['signal_date', 'entry_date'])
state_summary = pd.read_csv(RESEARCH_DIR / 'event_state_summary.csv')
transition_summary = pd.read_csv(RESEARCH_DIR / 'markov_transition_summary.csv')
model_metrics = pd.read_csv(RESEARCH_DIR / 'model_metrics.csv')
event_predictions = pd.read_csv(RESEARCH_DIR / 'event_model_predictions.csv', parse_dates=['signal_date'])
strategy_summary = pd.read_csv(RESEARCH_DIR / 'strategy_summary.csv')
strategy_periods = pd.read_csv(RESEARCH_DIR / 'strategy_periods.csv', parse_dates=['signal_date'])
panel = pd.read_csv(RESEARCH_DIR / 'risk_management_signal_panel.csv', parse_dates=['signal_date', 'entry_date'])

headline = pd.DataFrame({
    'metric': ['panel rows', 'extreme risk-off events', 'predicted events', 'hold days', 'core weight', 'tactical sleeve'],
    'value': [summary['rows'], summary['extreme_risk_off_events'], summary['prediction_rows'], summary['hold_days'], summary['core_weight'], summary['tactical_weight']],
})
display(headline)

## 2. Leakage controls first

If the leakage controls are not sound, nothing else matters. The research module uses four rules:
1. Features come only from current and trailing history.
2. The sleeve enters **one bar after** the signal date.
3. Markov and Bayesian probabilities are expanding-history estimates.
4. ML predictions are walk-forward and only evaluated out of sample.


In [ ]:
for guardrail in summary['leakage_guardrails']:
    print('-', guardrail)

## 3. The positive-drift problem

SPY rises more often than it falls over medium horizons. That means a tactical sleeve should not react to fear alone. It has to beat the positive drift that already exists in the baseline.

In [ ]:
display(drift)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
sns.barplot(data=drift, x='group', y='mean_return', hue='group', dodge=False, legend=False, palette='crest', ax=ax)
ax.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
ax.set_title('Forward drift by context group')
ax.set_xlabel('')
ax.set_ylabel('mean forward return')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Defining the extreme risk-off window

The sleeve only activates when market stress and macro fragility line up. The gate is deliberately strict because false positives are expensive against a market with positive drift.

In [ ]:
display(state_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
sns.barplot(data=state_summary, y='source_state', x='events', hue='source_state', dodge=False, legend=False, palette='rocket', ax=axes[0])
axes[0].set_title('Extreme risk-off states by count')
axes[0].set_xlabel('event count')
axes[0].set_ylabel('')

sns.barplot(data=state_summary, y='source_state', x='mean_event_return', hue='source_state', dodge=False, legend=False, palette='viridis', ax=axes[1])
axes[1].axvline(0.0, color='black', linestyle='--', linewidth=1.0)
axes[1].set_title('Average forward return by extreme state')
axes[1].set_xlabel('forward return over sleeve window')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 5. Markov chain view

The Markov lens asks: *when the market enters this extreme state, where does it tend to go next by the sleeve exit?* This is transition modelling, not prediction magic. The edge comes from asymmetry in the transition table.

In [ ]:
display(transition_summary.round(4))

heatmap = transition_summary.pivot(index='source_state', columns='future_sentiment_state', values='transition_count').fillna(0)
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.heatmap(heatmap, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
ax.set_title('Empirical transition counts from extreme risk-off states')
ax.set_xlabel('future sentiment state at sleeve exit')
ax.set_ylabel('current extreme state')
plt.tight_layout()
plt.show()

## 6. Bayesian shrinkage view

The Bayesian model takes the same state-level evidence but shrinks it toward the global instability baseline. This is the right mindset when event counts are small: do not trust a sparse state as much as a broad prior.

In [ ]:
posterior_cols = [
    'signal_date', 'bayesian_expected_return', 'bayesian_positive_prob', 'bayesian_training_mean_return', 'bayesian_training_positive_rate'
]
display(event_predictions[posterior_cols].dropna().head(12).round(4))

## 7. Walk-forward ML view

The ML block is intentionally modest. It predicts the tactical sleeve return from lagged SPY, VIX, credit, financial conditions, curve stress, policy uncertainty, and release-aware consumer sentiment.

In [ ]:
display(model_metrics.round(4))

metric_plot = model_metrics.loc[model_metrics['scope'].isin(['validation', 'holdout'])].copy()
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.2))
sns.barplot(data=metric_plot, x='model_name', y='roc_auc', hue='scope', ax=axes[0])
axes[0].axhline(0.5, color='black', linestyle='--', linewidth=1.0)
axes[0].set_title('Positive-return discrimination')
axes[0].set_xlabel('')
axes[0].set_ylabel('ROC AUC')

sns.barplot(data=metric_plot, x='model_name', y='top_quintile_mean_return', hue='scope', ax=axes[1])
axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
axes[1].set_title('Actual return in the top predicted quintile')
axes[1].set_xlabel('')
axes[1].set_ylabel('mean return')
plt.tight_layout()
plt.show()

## 8. Ensemble and tactical capital allocation

The final sleeve averages the Markov, Bayesian, and ML expected returns and probabilities. The sleeve only deploys if the ensemble beats the historical instability baseline. That is the core drift-aware risk-management rule.

In [ ]:
display(strategy_summary.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.4))
for strategy_name in ['core_60_only', 'full_spy_100', 'heuristic_panic_rebound', 'ensemble_mobilized', 'ensemble_long_short']:
    subset = strategy_periods.loc[strategy_periods['strategy_name'] == strategy_name].copy()
    if subset.empty:
        continue
    subset = subset.sort_values('signal_date')
    subset['equity'] = (1.0 + subset['strategy_return']).cumprod()
    ax.plot(subset['signal_date'], subset['equity'], linewidth=2.0, label=strategy_name)
ax.set_title('Equity curves: core, benchmark, heuristic, and ensemble sleeve')
ax.set_xlabel('signal date')
ax.set_ylabel('equity growth')
ax.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()

## 9. Where the model still fails

A sound model is not one that always makes money. It is one that shows you **where it fails** and why. If the tactical sleeve cannot reliably beat the drift baseline, the honest conclusion is that the sleeve is a risk-shaping tool, not a magic return engine.

In [ ]:
display(
    strategy_periods.loc[strategy_periods['strategy_name'].isin(['heuristic_panic_rebound', 'ensemble_mobilized', 'ensemble_long_short'])]
    .groupby('strategy_name')[['strategy_return', 'spy_return', 'excess_return_vs_spy', 'excess_return_vs_core']]
    .mean()
    .round(4)
)

recent_bad = strategy_periods.loc[
    (strategy_periods['strategy_name'] == 'ensemble_long_short')
    & (strategy_periods['excess_return_vs_spy'] < 0.0)
].sort_values('signal_date', ascending=False).head(12)
display(recent_bad.round(4))

## 10. Conclusions

**Read this as a portfolio engineer, not as a storyteller.**
1. The 60% core is the anchor because SPY drift is too strong to abandon lightly.
2. The 40% sleeve is justified only when instability creates an edge larger than the drift baseline.
3. Markov transitions help with path context, Bayesian shrinkage prevents overreacting to sparse states, and ML helps when feature interactions matter.
4. If the ensemble improves drawdown or instability capture without beating 100% SPY on terminal wealth, that is still useful risk management.
5. The next step is not more narrative. It is better execution logic, better features, and cleaner event definitions.
